# Caderno de aprendizagem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/00_aprendizagem.ipynb)

Este caderno vem **antes** dos notebooks do TCC (01 a 09). Os notebooks do TCC são a pesquisa em si: rodam o
modelo para os 50 municípios, calibram o ky e particionam o yield gap. Aqui você aprende, em exemplos pequenos, cada
ideia e cada ferramenta que eles usam, para chegar lá sabendo o que está acontecendo.

### Como o caderno está organizado

Um capítulo para cada notebook do TCC, na mesma ordem. Cada capítulo tem dois tipos de item:

- **Resolvido**: o exemplo vem pronto, com a explicação antes e os comentários no código. Rode, leia e mude os
  valores para ver o que acontece;
- **Para pensar e resolver**: perguntas para responder com código ou com texto, na célula vazia logo abaixo.
  Algumas pedem uma conta, outras pedem só que você explique um resultado com suas palavras.

| Capítulo | Assunto | Notebook do TCC |
|---|---|---|
| 1 | Python para agrometeorologia: funções, laços, graus-dia | [01_python_basico](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/01_python_basico.ipynb) |
| 2 | Tabelas, séries de produtividade e tendência | [02_dados_ibge_clima](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/02_dados_ibge_clima.ipynb) |
| 3 | Balanço hídrico, ISNA e a equação do ky | [03_mza_um_ciclo](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/03_mza_um_ciclo.ipynb) |
| 4 | Muitas simulações com laços, gravar e retomar | [04_simulacao_oeste](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/04_simulacao_oeste.ipynb) |
| 5 | Comparar o simulado com o observado | [05_verificacao_pp](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/05_verificacao_pp.ipynb) |
| 6 | Erro, calibração e validação | [06_calibracao_ky](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/06_calibracao_ky.ipynb) |
| 7 | Partição do yield gap e tendência | [07_yield_gap](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/07_yield_gap.ipynb) |
| 8 | Modelo de processo e ky emergente | [08_dssat_ky_emergente](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/08_dssat_ky_emergente.ipynb) |
| 9 | Figuras e tabelas para o artigo | [09_figuras_artigo](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/09_figuras_artigo.ipynb) |

### Como usar

1. Faça **Arquivo → Salvar uma cópia no Drive** e trabalhe na cópia.
2. Rode a célula 0 e a célula de dados, e depois siga os capítulos em ordem. Os capítulos 4 e 5 usam resultados
   que você mesmo calcula no capítulo anterior.
3. Termine o caderno inteiro antes de abrir o notebook 01 do TCC. Leve as dúvidas para a reunião com o
   orientador: os itens "para pensar" são a base da conversa.

Nada deste caderno precisa ser guardado no Drive. Os resultados da pesquisa saem dos notebooks 01 a 09.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab.
# Este caderno não grava nada no Drive: o que ele salva fica na máquina do Colab e some no fim da sessão.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")

SAIDA = Path("resultados/aprendizagem")          # pasta de rascunho deste caderno
SAIDA.mkdir(parents=True, exist_ok=True)
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mza_oeste as M                    # as funções do projeto (pasta src/)

clima = pd.read_parquet("dados/clima_oeste.parquet")
mun = pd.read_csv("dados/municipios_oeste.csv").set_index("municipio")
ibge = pd.read_csv("dados/ibge_pam_milho_oeste.csv")
ibge = ibge[(ibge["safra"] == 2) & (ibge["area_colhida_ha"] >= 100)]     # só a segunda safra

def clima_de(nome):
    """Clima diário de um município, com a data como índice."""
    return clima[clima["cod_ibge"] == mun.loc[nome, "cod_ibge"]].set_index("data")

sh = mun.loc["Santa Helena"]
clima_sh = clima_de("Santa Helena")
print(f"Santa Helena: lat {sh['latitude']:.2f}, CAD {sh['dta_mm_m']:.0f} mm/m, {len(clima_sh)} dias de clima")

---

# Capítulo 1. Python para agrometeorologia

**Prepara para:** [01_python_basico](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/01_python_basico.ipynb)


O modelo do projeto decide em que fase o milho está pela **soma térmica**: os graus-dia acumulados desde a
semeadura. Este capítulo usa essa conta para treinar o básico de Python: variáveis, listas, laços, decisões e
funções.

### 1.1 Resolvido: graus-dia de uma semana

Os graus-dia (GD) medem o calor acima da temperatura-base (Tb = 10 °C para o milho):
GD = máx(Tmed − Tb; 0). A função recebe Tmax e Tmin e devolve os GD do dia. O laço `for` percorre os dias.

In [ ]:
def graus_dia(tmax, tmin, tb=10.0):
    tmed = (tmax + tmin) / 2
    return max(tmed - tb, 0.0)                    # abaixo de Tb a planta não se desenvolve

semana = clima_sh.loc["2019-02-05":"2019-02-11"]  # a primeira semana após a semeadura de 05/02/2019
soma = 0.0
for data, dia in semana.iterrows():               # iterrows percorre a tabela linha a linha
    gd = graus_dia(dia["Tmax"], dia["Tmin"])
    soma = soma + gd
    print(f"{data:%d/%m}  Tmax {dia['Tmax']:4.1f}  Tmin {dia['Tmin']:4.1f}  GD {gd:4.1f}  acumulado {soma:5.1f}")

### 1.2 Resolvido: quantos dias até a maturação

O híbrido do projeto atinge a maturação com 1.450 °C dia. O laço `while` repete **enquanto** a condição for
verdadeira: soma os GD dia a dia até passar da soma térmica.

In [ ]:
def dias_ate_maturacao(clima_mun, semeadura, ct=1450.0, tb=10.0):
    dia = pd.Timestamp(semeadura)
    soma, n = 0.0, 0
    while soma < ct:
        soma = soma + graus_dia(clima_mun.loc[dia, "Tmax"], clima_mun.loc[dia, "Tmin"], tb)
        dia = dia + pd.Timedelta(days=1)
        n = n + 1
    return n

n = dias_ate_maturacao(clima_sh, "2019-02-05")
print(f"Semeadura em 05/02/2019: maturação após {n} dias")

### 1.3 Para pensar e resolver

1. Calcule os dias até a maturação para as semeaduras de 15/01, 05/02 e 15/03 de 2019, com um laço sobre uma lista
   de datas. O ciclo fica mais curto ou mais longo quando a semeadura atrasa? Explique pelo clima do outono.
2. Refaça a conta de 05/02/2019 com Tb = 8 °C. Por que o ciclo muda, se o clima é o mesmo?
3. Escreva uma função `dias_frios(clima_mun, inicio, fim)` que conte os dias com Tmin ≤ 2 °C (risco de geada) entre
   duas datas. Teste em Santa Helena e em Cascavel (`clima_de("Cascavel")`) de maio a julho de 2021.

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 2. Tabelas, séries de produtividade e tendência

**Prepara para:** [02_dados_ibge_clima](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/02_dados_ibge_clima.ipynb)


A produtividade do IBGE sobe ao longo dos anos por genética e manejo (a **tendência tecnológica**). Para enxergar o
efeito do clima, a tendência precisa ser retirada. Este capítulo treina o pandas (filtrar, agrupar, juntar
tabelas) com a série de Santa Helena.

### 2.1 Resolvido: a série de um município e a tendência

`np.polyfit(x, y, 1)` ajusta uma reta y = a·x + b. A inclinação `a` é o ganho tecnológico, em kg/ha por ano.

In [ ]:
serie = ibge[ibge["municipio"] == "Santa Helena"].set_index("ano")["rendimento_kg_ha"]
a, b = np.polyfit(serie.index, serie.values, 1)
tendencia = a * serie.index + b

plt.plot(serie.index, serie.values, "o-", label="IBGE")
plt.plot(serie.index, tendencia, "--", label=f"tendência: {a:.0f} kg/ha por ano")
plt.ylabel("kg/ha"); plt.title("Santa Helena, milho segunda safra"); plt.legend(); plt.show()

### 2.2 Resolvido: a anomalia e os anos de quebra

Anomalia relativa = PR / tendência − 1. Um valor de −0,40 quer dizer 40% abaixo do esperado para aquele ano.
`sort_values` ordena a tabela.

In [ ]:
anomalia = (serie / tendencia - 1).rename("anomalia")
anomalia.sort_values().head(5).round(2)

### 2.3 Resolvido: a chuva da safrinha e a anomalia

`groupby` agrupa as linhas e resume cada grupo. Aqui, a chuva de fevereiro a junho de cada ano. `pd.concat` junta
as duas séries lado a lado, pelo ano.

In [ ]:
safrinha = clima_sh[clima_sh.index.month.isin([2, 3, 4, 5, 6])]
chuva = safrinha.groupby(safrinha.index.year)["pr"].sum().rename("chuva_fev_jun")
tab = pd.concat([anomalia, chuva], axis=1).dropna()      # só os anos que existem nas duas
print(f"Correlação chuva × anomalia: {tab.corr().iloc[0, 1]:.2f}")
tab.plot.scatter(x="chuva_fev_jun", y="anomalia"); plt.show()

### 2.4 Para pensar e resolver

1. Refaça 2.1 e 2.2 para Cascavel. O ganho tecnológico é parecido? Os piores anos são os mesmos?
2. A correlação de 2.3 está longe de 1. Cite duas razões agronômicas para isso (pense em **quando** a chuva cai e
   em outros riscos do outono).
3. Calcule a produtividade média da região em cada ano **ponderada pela área colhida**:
   `ibge.groupby("ano").apply(lambda g: np.average(g["rendimento_kg_ha"], weights=g["area_colhida_ha"]))`.
   Por que a média ponderada é diferente da média simples? Qual representa melhor a produção da região?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 3. Balanço hídrico, ISNA e a equação do ky

**Prepara para:** [03_mza_um_ciclo](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/03_mza_um_ciclo.ipynb)


O coração do projeto é a cadeia **água no solo → ETR/ETc (ISNA) → perda de produtividade pelo ky**. Aqui você faz
essa cadeia à mão, com um balanço hídrico simplificado e poucos dias, para depois reconhecer cada passo no modelo
completo.

### 3.1 Resolvido: um balanço hídrico de "balde"

O solo é um balde com capacidade CAD (mm). A cada dia entra a chuva, e a planta tenta tirar a ETc. Se a água do
balde mais a chuva não bastar, a ETR fica abaixo da ETc e há déficit. O que passa da CAD é excedente (drena).

Esta é a versão mais simples possível: no modelo do projeto (`balanco_hidrico_cultura`), a ETR cai aos poucos à
medida que o solo seca, e a CAD cresce com a raiz. A lógica é a mesma.

In [ ]:
def balde(chuva, etc, cad, arm_inicial):
    arm = arm_inicial
    linhas = []
    for p, demanda in zip(chuva, etc):            # zip percorre as duas listas juntas
        disponivel = arm + p
        etr = min(demanda, disponivel)            # a planta não tira mais do que há
        arm = disponivel - etr
        exc = max(arm - cad, 0.0)                 # o que passa da capacidade drena
        arm = arm - exc
        linhas.append({"chuva": p, "ETc": demanda, "ETR": etr, "DEF": demanda - etr, "ARM": arm, "EXC": exc})
    return pd.DataFrame(linhas)

chuva = [12, 0, 0, 0, 0, 0, 0, 25, 0, 0, 0, 0, 0, 0]       # mm, duas semanas
etc = [5.5] * 14                                             # mm/dia, milho em pleno florescimento
cad = 0.40 * 120                                             # raiz de 0,40 m × 120 mm/m = 48 mm
bh = balde(chuva, etc, cad, arm_inicial=30)
print(f"ISNA = ETR/ETc = {bh['ETR'].sum() / bh['ETc'].sum():.2f}")
bh.round(1)

### 3.2 Resolvido: do ISNA à produtividade atingível

Para cada fase *i*, a fração perdida é ky,i × (1 − ISNA,i). Na forma do **produtório**, as perdas se multiplicam:

$$PA = PP \times \prod_{i} \left[1 - k_{y,i}\left(1 - ISNA_i\right)\right]$$

Os ISNA abaixo são os de Santa Helena em 2019 (notebook 03). `M.KY_DK` guarda os ky de Doorenbos & Kassam.

In [ ]:
isna = {"estabelecimento": 0.85, "vegetativo": 0.85, "floracao": 0.83, "enchimento": 0.60, "maturacao": 0.98}
pp = 6100                                                    # kg/ha

fator = 1.0
for fase, ky in M.KY_DK.items():
    perda = ky * (1 - isna[fase])
    fator = fator * max(1 - perda, 0)                        # nunca negativo
    print(f"{fase:16s} ky {ky:.1f}  ISNA {isna[fase]:.2f}  perda {perda:.2f}  fator acumulado {fator:.3f}")
print(f"PA = {pp * fator:.0f} kg/ha")

### 3.3 Para pensar e resolver

1. Com ky = 1,5 na floração, qual ISNA dessa fase zera a PA, mesmo com as outras fases sem déficit? Faça a conta
   no papel e confira com código.
2. No balde de 3.1, troque a CAD para a de uma raiz de 1,0 m (120 mm). Como mudam a ETR e o ISNA? O que isso diz
   sobre a profundidade de raiz usada no modelo (0,40 m)?
3. Refaça 3.2 com ky da floração = 0,8. Quanto a PA sobe? Guarde este resultado: é o tipo de mudança que a
   calibração do capítulo 6 vai buscar.

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 4. Muitas simulações com laços, gravar e retomar

**Prepara para:** [04_simulacao_oeste](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/04_simulacao_oeste.ipynb)


O notebook 04 roda 12.150 ciclos (50 municípios × 27 anos × 9 datas). Aqui você roda o mesmo modelo para poucos
casos, junta os resultados numa tabela e aprende a gravar e retomar, que é o que protege a simulação grande quando
o Colab cai.

### 4.1 Resolvido: um laço duplo de simulações

`M.simular` roda um ciclo e devolve um dicionário. Uma lista de dicionários vira uma tabela com `pd.DataFrame`.

In [ ]:
import time

inicio = time.time()
linhas = []
for ano in [2019, 2020, 2021]:
    for mes, dia in M.DATAS:                                  # as 9 datas de semeadura do projeto
        r = M.simular(clima_sh, sh["latitude"], sh["dta_mm_m"], pd.Timestamp(ano, mes, dia))
        linhas.append({"ano": ano, "semeadura": f"{dia:02d}/{mes:02d}", "PP": r["PPf"], "PA": r["PA"],
                       "ciclo_dias": r["ciclo_dias"]})
seg = time.time() - inicio
sim4 = pd.DataFrame(linhas)
print(f"{len(sim4)} ciclos em {seg:.1f} s ({seg / len(sim4):.2f} s por ciclo)")
sim4.head()

### 4.2 Resolvido: resumir por grupo

`pivot_table` monta uma tabela cruzada: aqui, a PP média por data de semeadura e ano.

In [ ]:
sim4.pivot_table(index="semeadura", columns="ano", values="PP", sort=False).round(0)

### 4.3 Resolvido: gravar e retomar

Cada ano é gravado num arquivo próprio. Se o arquivo já existe, o laço pula aquele ano. Rode a célula duas vezes:
na segunda, nada é simulado de novo.

In [ ]:
parcial = SAIDA / "parcial"
parcial.mkdir(exist_ok=True)
for ano, g in sim4.groupby("ano"):
    arquivo = parcial / f"santa_helena_{ano}.parquet"
    if arquivo.exists():
        print(ano, "já gravado: pula")
        continue
    g.to_parquet(arquivo)
    print(ano, "gravado")

todos = pd.concat([pd.read_parquet(a) for a in sorted(parcial.glob("*.parquet"))])   # junta os arquivos
print(len(todos), "linhas lidas de volta")

### 4.4 Para pensar e resolver

1. Com o tempo por ciclo de 4.1, estime quanto tempo levariam os 12.150 ciclos do notebook 04.
2. Acrescente um laço externo sobre os municípios `["Santa Helena", "Cascavel", "Guaíra"]` (use `clima_de` e
   `mun.loc[nome]`), só para 2019. Em qual município a PA é maior? Relacione com a CAD do solo (`dta_mm_m`).
3. Por que a PP cai das semeaduras de janeiro para as de março? Responda olhando a radiação (`Rs`) e a temperatura
   de março a junho.

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 5. Comparar o simulado com o observado

**Prepara para:** [05_verificacao_pp](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/05_verificacao_pp.ipynb)


A PP é o teto: com manejo perfeito e sem falta de água, nenhum produtor deveria colher acima dela de forma
sistemática. Este capítulo ensina a confrontar uma simulação com dados observados: juntar as tabelas, contar
casos, usar quantis e fazer o gráfico 1:1.

### 5.1 Resolvido: a PP da janela de semeadura × o IBGE

A janela de semeadura do produtor tem 4 datas (`M.JANELA`). A PP comparável ao IBGE é a média dessas datas. O laço
roda 4 datas × 23 anos de Santa Helena (poucos segundos).

In [ ]:
linhas = []
for ano in range(2003, 2026):
    for data in M.JANELA:                                     # "15/01", "05/02", ...
        dia, mes = map(int, data.split("/"))
        r = M.simular(clima_sh, sh["latitude"], sh["dta_mm_m"], pd.Timestamp(ano, mes, dia))
        linhas.append({"ano": ano, "PP": r["PPf"], "PA": r["PA"]})
janela = pd.DataFrame(linhas).groupby("ano").mean()          # média das 4 datas, por ano
comp = janela.join(serie.rename("PR"), how="inner")          # junta com o IBGE pelo ano
comp["PR_sobre_PP"] = comp["PR"] / comp["PP"]
print(f"Anos com PR > PP: {(comp['PR'] > comp['PP']).sum()} de {len(comp)}")
comp.round(2).tail()

### 5.2 Resolvido: o gráfico 1:1

Num gráfico 1:1, os pontos acima da diagonal são anos em que o observado passou do simulado.

In [ ]:
lim = [0, 10000]
plt.figure(figsize=(5, 5))
plt.scatter(comp["PP"], comp["PR"])
plt.plot(lim, lim, "k--", label="1:1")
plt.xlim(lim); plt.ylim(lim)
plt.xlabel("PP simulada (kg/ha)"); plt.ylabel("PR do IBGE (kg/ha)"); plt.legend(); plt.show()

### 5.3 Resolvido: quantis

O quantil 0,95 é o valor que fica acima de 95% dos dados. Ele é usado no notebook 05 para propor quanto a PP deveria
subir para ficar acima de quase todas as PR.

In [ ]:
q95 = np.quantile(comp["PR_sobre_PP"], 0.95)
print(f"PR/PP: mediana {comp['PR_sobre_PP'].median():.2f}, quantil 0,95 {q95:.2f}")

### 5.4 Para pensar e resolver

1. A PP é proporcional ao índice de colheita (Cc = 0,40 no projeto). Se a PP for multiplicada pelo quantil 0,95 de
   5.3, a que Cc isso equivale? É um valor plausível para híbridos atuais?
2. Repita 5.3 só com os anos de 2015 em diante. O fator muda? Por que os anos recentes pesam mais nessa decisão?
3. Explique, com suas palavras, por que uma PP abaixo da PR em muitos anos invalidaria a partição do yield gap.

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 6. Erro, calibração e validação

**Prepara para:** [06_calibracao_ky](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/06_calibracao_ky.ipynb)


Calibrar é procurar o valor do parâmetro que faz o modelo reproduzir o observado. Para aprender sem depender da
simulação grande, este capítulo usa **dados sintéticos**: você cria os dados com um ky conhecido e confere se a
calibração o encontra. É assim que se testa um método antes de confiar nele.

### 6.1 Resolvido: os índices de desempenho

O arquivo `src/avaliacao.py` tem os índices usados no projeto. O RMSE está na unidade dos dados; o NSE vai de −∞ a 1
(0 = tão bom quanto usar a média).

In [ ]:
from avaliacao import avaliar, rmse

obs = np.array([0.95, 0.80, 0.60, 1.00, 0.70])
sim = np.array([0.90, 0.85, 0.50, 0.97, 0.78])
avaliar(obs, sim).round(3)

### 6.2 Resolvido: dados sintéticos com um ky conhecido

Trinta "anos" com o ISNA da floração sorteado entre 0,4 e 1,0. A produtividade relativa observada segue a equação
do ky com **ky verdadeiro = 0,8**, mais um ruído (a parte que o modelo não explica).

In [ ]:
rng = np.random.default_rng(42)                               # semente: o sorteio sai igual toda vez
anos = np.arange(1996, 2026)
isna_flor = rng.uniform(0.4, 1.0, len(anos))
KY_VERDADEIRO = 0.8
rel_obs = np.clip(1 - KY_VERDADEIRO * (1 - isna_flor) + rng.normal(0, 0.05, len(anos)), 0, 1)

def rel_sim(ky, isna):
    return np.clip(1 - ky * (1 - isna), 0, 1)

print(f"RMSE com ky = 1,5 (tabelado): {rmse(rel_obs, rel_sim(1.5, isna_flor)):.3f}")
print(f"RMSE com ky = 0,8 (verdadeiro): {rmse(rel_obs, rel_sim(0.8, isna_flor)):.3f}")

### 6.3 Resolvido: busca em grade e otimizador

A **grade** testa todos os valores numa malha e permite ver a curva do erro. O **otimizador**
(`scipy.optimize.minimize`) parte de um chute e caminha "morro abaixo". Se os dois concordam, a calibração é
confiável.

In [ ]:
from scipy.optimize import minimize

grade = np.round(np.arange(0.1, 2.01, 0.05), 2)
erros = [rmse(rel_obs, rel_sim(k, isna_flor)) for k in grade]       # list comprehension: um laço numa linha
ky_grade = grade[np.argmin(erros)]

res = minimize(lambda x: rmse(rel_obs, rel_sim(x[0], isna_flor)), x0=[1.5], bounds=[(0.05, 2.5)])
print(f"Grade: ky = {ky_grade:.2f}   Otimizador: ky = {res.x[0]:.2f}   (verdadeiro: {KY_VERDADEIRO})")

plt.plot(grade, erros); plt.axvline(ky_grade, ls="--", color="r")
plt.xlabel("ky floração"); plt.ylabel("RMSE"); plt.title("Curva do erro"); plt.show()

### 6.4 Resolvido: validação deixando um ano de fora

Para cada ano, calibra-se com os outros 29 e testa-se no ano retirado. A dispersão dos ky calibrados é a
**incerteza** do ky, e o erro nos anos retirados é o desempenho honesto do modelo.

In [ ]:
kys, previstos = [], []
for i in range(len(anos)):
    fica = np.arange(len(anos)) != i                          # todos menos o ano i
    r = minimize(lambda x: rmse(rel_obs[fica], rel_sim(x[0], isna_flor[fica])), x0=[1.5], bounds=[(0.05, 2.5)])
    kys.append(r.x[0])
    previstos.append(rel_sim(r.x[0], isna_flor[i]))
kys = np.array(kys)
print(f"ky: média {kys.mean():.2f}, faixa 5-95%: {np.quantile(kys, 0.05):.2f} a {np.quantile(kys, 0.95):.2f}")
avaliar(rel_obs, np.array(previstos)).round(3)

### 6.5 Para pensar e resolver

1. **O raciocínio circular.** Suponha que o produtor colhe só 70% da PA por causa do manejo:
   `pr = 0.7 * rel_obs * 9000` (kg/ha, com PP = 9.000). Calibre o ky comparando `pr` com a PA simulada
   `9000 * rel_sim(ky, isna_flor)`. Que ky sai? Agora calibre com a produtividade **relativa** `pr / pr.max()`
   contra `rel_sim(ky, isna_flor) / rel_sim(ky, isna_flor).max()`. Qual das duas recupera o ky verdadeiro? Por
   quê? (Essa é a razão das estratégias A e B do notebook 06.)
2. Aumente o ruído de 0,05 para 0,15 e refaça 6.3 e 6.4. O que acontece com a faixa do ky?
3. Crie um segundo ISNA (enchimento) **parecido** com o da floração (`isna_ench = isna_flor + rng.normal(0, 0.05, 30)`)
   e dados com os dois ky (0,8 e 0,5). Faça a grade em duas dimensões e o gráfico com `plt.contourf`. Por que
   aparece um "vale" comprido, com várias combinações de erro parecido?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 7. Partição do yield gap e tendência

**Prepara para:** [07_yield_gap](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/07_yield_gap.ipynb)


Com a PP, a PA e a PR em mãos, o yield gap é dividido em três parcelas. Este capítulo faz a partição com números
simples, mostra por que há um limite na parcela da data de semeadura e ensina os testes de tendência.

### 7.1 Resolvido: a partição de um município-ano

As equações (2) a (7) do item 4.5 do projeto, escritas em código:

```
YG_total      = PP_janela − PR
YG_hídrico    = PP_janela − PA_janela
YG_data       = mín[máx(PA_melhor − PA_janela; 0); YG_hídrico]
YG_inevitável = YG_hídrico − YG_data
YG_manejo     = PA_janela − PR
YG_explorável = 0,8 × PA_janela − PR
```

In [ ]:
def particao(pp_janela, pa_janela, pa_melhor, pr):
    yg_total = pp_janela - pr
    yg_hidrico = pp_janela - pa_janela
    yg_data = np.minimum(np.maximum(pa_melhor - pa_janela, 0), yg_hidrico)
    return pd.DataFrame({"YG_total": yg_total, "YG_inevitavel": yg_hidrico - yg_data, "YG_data": yg_data,
                         "YG_manejo": pa_janela - pr, "YG_exploravel": 0.8 * pa_janela - pr})

p = particao(pp_janela=np.array([9000]), pa_janela=np.array([6500]), pa_melhor=np.array([7200]), pr=np.array([5000]))
print("Soma das três parcelas:", (p["YG_inevitavel"] + p["YG_data"] + p["YG_manejo"]).iloc[0], "= YG_total")
p

### 7.2 Resolvido: a mesma função para muitas linhas

`np.minimum` e `np.maximum` trabalham elemento a elemento, então a função serve para uma tabela inteira. No segundo
caso a melhor data é pior que a janela (sem ganho com a data); no terceiro, o produtor colhe acima da PA.

In [ ]:
casos = pd.DataFrame({"PP_janela": [9000, 9000, 9000], "PA_janela": [6500, 7000, 5000],
                      "PA_melhor": [7200, 6800, 5600], "PR": [5000, 5000, 5400]})
casos.join(particao(casos["PP_janela"], casos["PA_janela"], casos["PA_melhor"], casos["PR"]))

### 7.3 Resolvido: Mann-Kendall e inclinação de Sen

O **teste de Mann-Kendall** pergunta se uma série tende a subir ou descer, sem supor uma reta: é a correlação de
Kendall entre o tempo e os valores. A **inclinação de Sen** é a mediana das inclinações entre todos os pares de
anos, pouco sensível a anos extremos. Aqui, na série do IBGE de Santa Helena.

In [ ]:
from scipy.stats import kendalltau, theilslopes

tau, p_valor = kendalltau(serie.index, serie.values)
sen, intercepto, sen_baixo, sen_alto = theilslopes(serie.values, serie.index)
print(f"Mann-Kendall: tau = {tau:.2f}, p = {p_valor:.4f}")
print(f"Sen: {sen:.0f} kg/ha por ano (IC 95%: {sen_baixo:.0f} a {sen_alto:.0f});  polyfit do cap. 2: {a:.0f}")

### 7.4 Para pensar e resolver

1. No terceiro caso de 7.2, o YG_manejo sai negativo. O que isso quer dizer agronomicamente? Que erro do modelo ou
   dos dados pode causar isso?
2. Por que o YG_data precisa ser limitado ao YG_hídrico? Monte um caso em que, sem o limite, o YG_inevitável sairia
   negativo.
3. Retire o ano de 2021 da série e refaça 7.3. Quanto mudam a inclinação de Sen e a do `polyfit`? Qual das duas é
   mais robusta a um ano extremo?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 8. Modelo de processo e ky emergente

**Prepara para:** [08_dssat_ky_emergente](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/08_dssat_ky_emergente.ipynb)


O DSSAT CERES-Maize simula a planta por processos (fotossíntese, crescimento de raízes, água em camadas do solo).
Rodando o mesmo clima em sequeiro e irrigado, a perda pelo déficit sai da razão entre os dois. O **ky emergente** é
o ky que, aplicado ao ISNA do MZA, reproduz essa perda. Aqui o DSSAT não roda: os dados são sintéticos, para
aprender a conta.

### 8.1 Resolvido: ajustar o ky emergente

Perda = 1 − sequeiro/irrigado. Se perda = ky × (1 − ISNA), o ky é a inclinação de uma reta que passa pela origem,
estimada por mínimos quadrados: ky = Σ(x·y) / Σ(x²), com x = 1 − ISNA e y = perda.

In [ ]:
irrigado = rng.normal(11500, 400, 40)                          # kg/ha, 40 ciclos
isna_ciclo = rng.uniform(0.5, 1.0, 40)
sequeiro = irrigado * (1 - 0.3 * (1 - isna_ciclo)) + rng.normal(0, 300, 40)

x = 1 - isna_ciclo
y = 1 - sequeiro / irrigado
ky_emergente = np.sum(x * y) / np.sum(x ** 2)
print(f"Perda média segundo o processo: {y.mean():.0%};  ky emergente = {ky_emergente:.2f}")

plt.scatter(x, y); plt.plot([0, 0.5], [0, 0.5 * ky_emergente], "r--")
plt.xlabel("1 − ISNA (MZA)"); plt.ylabel("perda = 1 − sequeiro/irrigado"); plt.show()

### 8.2 Resolvido: coeficientes de cultivar e soma térmica

No DSSAT, P1 e P5 são somas térmicas que controlam a duração das fases. A ideia é a mesma da soma térmica do MZA
(1.450 °C dia). Veja o efeito no ciclo com a função do capítulo 1.

In [ ]:
for ct in [1300, 1450, 1600]:
    print(f"CT = {ct} °C dia -> {dias_ate_maturacao(clima_sh, '2019-02-05', ct=ct)} dias até a maturação")

### 8.3 Para pensar e resolver

1. No notebook 08, a perda média segundo o DSSAT é de 9%, contra 70% no MZA com o ky tabelado. Se o DSSAT perde
   pouco onde o MZA diz que o ISNA é baixo, o ky emergente sobe ou desce? Confira mudando o 0,3 de 8.1 para 0,1.
2. O perfil de solo do DSSAT tem 180 cm; a raiz do MZA chega a 0,40 m. Use o balde do capítulo 3 para explicar por
   que isso muda a perda.
3. Para atrasar o florescimento de uma cultivar no DSSAT, você aumentaria ou diminuiria o P1? Por quê?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 9. Figuras e tabelas para o artigo

**Prepara para:** [09_figuras_artigo](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/09_figuras_artigo.ipynb)


Uma figura de artigo precisa ser lida sem o texto: eixos com nome e unidade, letras legíveis, legenda clara e
resolução de 300 dpi. Este capítulo mostra a diferença entre um gráfico rápido e uma figura pronta.

### 9.1 Resolvido: uma figura com dois painéis

`plt.subplots(1, 2)` cria dois painéis lado a lado. `savefig(dpi=300)` grava em alta resolução.

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.plot(serie.index, serie.values, "o-", color="0.2", ms=4, label="IBGE")
ax1.plot(serie.index, tendencia, "--", color="tab:red", label="tendência")
ax1.set_xlabel("Ano"); ax1.set_ylabel("Produtividade (kg ha⁻¹)"); ax1.legend(frameon=False)
ax1.text(0.02, 0.95, "(a)", transform=ax1.transAxes, fontweight="bold")

ax2.scatter(comp["PP"], comp["PR"], color="0.2", s=18)
ax2.plot(lim, lim, "k--", lw=0.8)
ax2.set_xlim(lim); ax2.set_ylim(lim)
ax2.set_xlabel("PP simulada (kg ha⁻¹)"); ax2.set_ylabel("PR observada (kg ha⁻¹)")
ax2.text(0.02, 0.95, "(b)", transform=ax2.transAxes, fontweight="bold")

fig.tight_layout()
fig.savefig(SAIDA / "fig_exemplo.png", dpi=300, bbox_inches="tight")
plt.show()

### 9.2 Resolvido: uma tabela para o artigo

Arredonde com a precisão que o artigo usa (kg/ha sem casas, razões com duas) e grave em CSV, que abre no Excel e no
Google Planilhas.

In [ ]:
tabela = comp[["PP", "PA", "PR"]].describe().loc[["mean", "std", "min", "max"]].round(0)
tabela.index = ["Média", "Desvio-padrão", "Mínimo", "Máximo"]
tabela.to_csv(SAIDA / "tabela_exemplo.csv")
tabela

### 9.3 Para pensar e resolver

1. Refaça, no padrão de 9.1, a figura da PP e da PA média por data de semeadura com os resultados do capítulo 4
   (`sim4`). Nomeie os eixos com unidade.
2. Abra `fig_exemplo.png` (no Colab, no ícone de pasta à esquerda, em `resultados/aprendizagem`). O que você
   mudaria para a figura caber numa coluna de revista (cerca de 8 cm de largura)?

---

## Ao terminar o caderno

Confira se você consegue explicar, sem olhar o código:

- o que são graus-dia e por que a semeadura tardia alonga o ciclo;
- o que é o ISNA e como o ky transforma o ISNA em perda de produtividade;
- por que a PP precisa ficar acima da PR;
- por que calibrar o ky contra a média do IBGE é circular;
- o que cada parcela do yield gap representa.

Leve as respostas dos itens "para pensar" para a reunião com o orientador. Depois, siga para o
[notebook 01 do TCC](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/01_python_basico.ipynb).

In [ ]:
# escreva sua resposta aqui